# Sprint 2 — Preprocessing Pipeline
**Input:** `data/cleaned/` (from `01_image_cleaning.ipynb`)  
**Output:** `train_loader`, `val_loader`, `test_loader` — ready for model training

In [1]:
pip install torch torchvision

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [3]:
CLEANED_DIR   = Path("./data/cleaned")
IMAGE_SIZE    = (224, 224)          # required input size
BATCH_SIZE    = 32
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

In [4]:

train_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),        # handles left/right hand variation
    transforms.RandomRotation(15),            # handles tilted hand positions
    transforms.ColorJitter(brightness=0.3, contrast=0.3),  # handles lighting variation
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_test_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [5]:
class ASLDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir      = Path(root_dir)
        self.transform     = transform
        self.classes       = sorted([d.name for d in self.root_dir.iterdir() if d.is_dir()])
        self.class_to_idx  = {c: i for i, c in enumerate(self.classes)}
        valid_exts         = {".jpg", ".jpeg", ".png"}
        self.samples       = [
            (p, self.class_to_idx[cls])
            for cls in self.classes
            for p in (self.root_dir / cls).iterdir()
            if p.suffix.lower() in valid_exts
        ]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

In [6]:
train_loader = DataLoader(ASLDataset(CLEANED_DIR, train_transforms),
                          batch_size=BATCH_SIZE, shuffle=True)

val_loader   = DataLoader(ASLDataset(CLEANED_DIR, val_test_transforms),
                          batch_size=BATCH_SIZE, shuffle=False)

test_loader  = DataLoader(ASLDataset(CLEANED_DIR, val_test_transforms),
                          batch_size=BATCH_SIZE, shuffle=False)

print(f"Classes  : {train_loader.dataset.classes}")
print(f"Train    : {len(train_loader.dataset):,} images")
print(f"Val      : {len(val_loader.dataset):,} images")
print(f"Test     : {len(test_loader.dataset):,} images")

Classes  : ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'asl_alphabet_train', 'del', 'nothing', 'space']
Train    : 87,000 images
Val      : 87,000 images
Test     : 87,000 images


In [7]:
from pathlib import Path

print(CLEANED_DIR)
print(Path(CLEANED_DIR).exists())
print(list(Path(CLEANED_DIR).iterdir())[:10])

data/cleaned
True
[PosixPath('data/cleaned/R'), PosixPath('data/cleaned/U'), PosixPath('data/cleaned/I'), PosixPath('data/cleaned/N'), PosixPath('data/cleaned/G'), PosixPath('data/cleaned/Z'), PosixPath('data/cleaned/T'), PosixPath('data/cleaned/S'), PosixPath('data/cleaned/A'), PosixPath('data/cleaned/F')]


In [8]:
from pathlib import Path

image_extensions = ["*.jpg", "*.jpeg", "*.png"]

image_files = []
for ext in image_extensions:
    image_files.extend(Path(CLEANED_DIR).rglob(ext))

print("Total images found:", len(image_files))
print(image_files[:5])

Total images found: 87000
[PosixPath('data/cleaned/R/R2837.jpg'), PosixPath('data/cleaned/R/R2189.jpg'), PosixPath('data/cleaned/R/R1480.jpg'), PosixPath('data/cleaned/R/R1494.jpg'), PosixPath('data/cleaned/R/R2823.jpg')]
